# House Price Prediction — End-to-End CRISP-DM
## Professional Google Colab Notebook

**Objective:** build, evaluate, and productionize a defensible residential house-price model from the supplied `House Price.csv` dataset.

This notebook consolidates the full workflow completed in the earlier analysis:

1. Business understanding and analytical success criteria  
2. Data understanding, EDA, visualization, and leakage detection  
3. Deterministic cleaning and semantic missingness  
4. Leakage-safe preprocessing and feature engineering  
5. Outlier, anomaly, leverage, and influence analysis  
6. Multi-method feature selection  
7. Target-free property clustering  
8. Dummy, linear, regularized, robust, ensemble, and boosting regression  
9. Temporal validation and one-time future-holdout evaluation  
10. Production refit, inference service, review flags, monitoring, and export

> **Target clarification:** the dataset contains `price`, not household income. The supervised target is therefore **property sale price**.

> **Critical leakage rule:** `price_per_sqft` is excluded from all predictive models because it is calculated from the target: `price / sqft_living`.

### Recommended execution mode

- Keep `FAST_MODE = True` for a practical Colab run with the complete methodology and a compact compute budget.
- Set `FAST_MODE = False` for more resamples, more trees, and fuller diagnostic repetition.
- Use **Runtime → Run all** after uploading the CSV.

## 0. Notebook controls and reproducibility

The notebook uses fixed random seeds and time-ordered validation. All learned preprocessing is fitted inside each training fold. The final holdout is chronologically later than the development data.

In [ ]:
# Core runtime controls
import os

# In Colab, leave HOUSE_PRICE_RUN_MODE unset for the recommended 'fast' mode.
# Supported values: 'fast', 'full', and 'smoke' (the last is only for code validation).
RUN_MODE = os.environ.get('HOUSE_PRICE_RUN_MODE', 'fast').strip().lower()
if RUN_MODE not in {'fast', 'full', 'smoke'}:
    raise ValueError("HOUSE_PRICE_RUN_MODE must be 'fast', 'full', or 'smoke'.")
SMOKE_MODE = RUN_MODE == 'smoke'
FAST_MODE = RUN_MODE != 'full'
RANDOM_STATE = 42
AUTO_DOWNLOAD_RESULTS = False  # Set True in Colab to download the final ZIP automatically.

# Compute-conscious settings
N_CLUSTER_RESAMPLES = 1 if SMOKE_MODE else (5 if FAST_MODE else 20)
N_BOOTSTRAPS = 20 if SMOKE_MODE else (300 if FAST_MODE else 5000)
N_PERMUTATION_REPEATS = 1 if SMOKE_MODE else (2 if FAST_MODE else 5)
N_TREES = 8 if SMOKE_MODE else (20 if FAST_MODE else 100)
N_IFOREST_TREES = 20 if SMOKE_MODE else (100 if FAST_MODE else 300)

print({
    'RUN_MODE': RUN_MODE,
    'FAST_MODE': FAST_MODE,
    'cluster_resamples': N_CLUSTER_RESAMPLES,
    'bootstraps': N_BOOTSTRAPS,
    'trees_per_ensemble': N_TREES,
})

In [ ]:
# Imports and environment setup
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import joblib
from joblib.externals import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import sparse
from scipy.stats import rankdata

from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.covariance import MinCovDet
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
)
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, SplineTransformer, StandardScaler

warnings.filterwarnings('ignore', category=UserWarning, message='Found unknown categories.*')
warnings.filterwarnings('ignore', category=FutureWarning)

np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

IN_COLAB = 'google.colab' in sys.modules
default_output_dir = '/content/house_price_crispdm_outputs' if IN_COLAB else '/mnt/data/house_price_crispdm_outputs'
OUTPUT_DIR = Path(os.environ.get('HOUSE_PRICE_OUTPUT_DIR', default_output_dir))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
MODEL_DIR = OUTPUT_DIR / 'models'
for directory in (FIG_DIR, TABLE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)
print('Output directory:', OUTPUT_DIR)

## 1. Load the dataset

The cell looks for `House Price.csv` in common Colab and local-runtime locations. If it is not found in Colab, a file-upload dialog appears.

In [ ]:
def locate_or_upload_csv() -> Path:
    candidates = [
        Path('/content/House Price.csv'),
        Path('/content/House_Price.csv'),
        Path('/mnt/data/House Price.csv'),
        Path.cwd() / 'House Price.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_names:
            raise FileNotFoundError('Upload a CSV file named House Price.csv.')
        return Path('/content') / csv_names[0]

    raise FileNotFoundError('House Price.csv was not found.')

DATA_PATH = locate_or_upload_csv()
raw_source = pd.read_csv(DATA_PATH)
raw_source.columns = raw_source.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)

required_source_columns = {
    'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
    'floors', 'waterfront', 'view', 'condition', 'sqft_above',
    'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
    'statezip', 'price_per_sqft',
}
missing = sorted(required_source_columns - set(raw_source.columns))
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

file_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('Loaded:', DATA_PATH)
print('Shape:', raw_source.shape)
print('SHA-256:', file_hash)
display(raw_source.head())

# CRISP-DM Phase 1 — Business Understanding

## 1.1 Business problem

Estimate the likely sale price of a residential property using information available at valuation time.


a supervised regression function is learned:

\[
\widehat{Price}=f(Size, Location, Age, Condition, Amenities, \ldots)
\]

## 1.2 Unit of analysis

One row is treated as one recorded property transaction. Repeated addresses are retained because they may represent separate units or transactions.

## 1.3 Success criteria

The primary metric is **Mean Absolute Error**:

\[
MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|
\]

Secondary metrics include RMSE, median absolute error, \(R^2\), RMSLE, WAPE, signed error, and segment-level MAE.

## 1.4 Baselines

Every model must outperform:

- Median-price dummy baseline
- Mean-price dummy baseline
- Living-area-only linear regression

## 1.5 Scope limitation

The source transactions are from a short 2014 Washington State period. Any production use for a later market requires retraining on recent, representative transactions.

# CRISP-DM Phase 2 — Data Understanding
## 2.1 Structural audit and leakage detection

In [ ]:
source = raw_source.copy()
source.insert(0, 'source_row_id', np.arange(1, len(source) + 1, dtype=np.int64))
source['sale_date'] = pd.to_datetime(source['date'], errors='coerce')

expected_ppsf = source['price'] / source['sqft_living']
address_key = (
    source['street'].astype(str).str.strip().str.casefold() + '|' +
    source['city'].astype(str).str.strip().str.casefold() + '|' +
    source['statezip'].astype(str).str.strip().str.casefold()
)
address_counts = address_key.value_counts()

initial_audit = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'date_min', 'date_max', 'unique_dates',
        'explicit_missing_cells', 'exact_duplicate_rows', 'zero_price_rows',
        'negative_price_rows', 'zero_bedroom_rows', 'zero_bathroom_rows',
        'unique_streets', 'unique_cities', 'unique_statezip',
        'waterfront_rows', 'renovation_zero_sentinel_rows',
        'renovation_before_build_rows', 'repeated_address_groups',
        'rows_in_repeated_address_groups', 'area_identity_violations',
        'price_per_sqft_identity_violations',
    ],
    'value': [
        len(source), source.shape[1], source['sale_date'].min().date(),
        source['sale_date'].max().date(), source['sale_date'].nunique(),
        int(source.isna().sum().sum()), int(source.duplicated(subset=raw_source.columns).sum()),
        int(source['price'].eq(0).sum()), int(source['price'].lt(0).sum()),
        int(source['bedrooms'].eq(0).sum()), int(source['bathrooms'].eq(0).sum()),
        source['street'].nunique(), source['city'].nunique(), source['statezip'].nunique(),
        int(source['waterfront'].eq(1).sum()), int(source['yr_renovated'].eq(0).sum()),
        int(((source['yr_renovated'] > 0) & (source['yr_renovated'] < source['yr_built'])).sum()),
        int(address_counts.gt(1).sum()), int(address_counts[address_counts.gt(1)].sum()),
        int((source['sqft_living'] != source['sqft_above'] + source['sqft_basement']).sum()),
        int((~np.isclose(source['price_per_sqft'], expected_ppsf, atol=0.011, rtol=0)).sum()),
    ],
})
initial_audit.to_csv(TABLE_DIR / '01_initial_data_audit.csv', index=False)
display(initial_audit)

assert (source['sqft_living'] == source['sqft_above'] + source['sqft_basement']).all()
assert np.isclose(source['price_per_sqft'], expected_ppsf, atol=0.011, rtol=0).all()
print('Leakage finding: price_per_sqft is target-derived and will never enter X.')

## 2.2 Lock the temporal development and future-holdout periods

The split keeps complete dates together. Detailed EDA and all model choices are made on the earlier development period. The later period is reserved for one-time final evaluation.

In [ ]:
positive_source = source.loc[source['price'] > 0].sort_values(['sale_date', 'street', 'source_row_id']).copy()
daily_counts = positive_source.groupby('sale_date').size().sort_index()
cumulative_share = daily_counts.cumsum() / len(positive_source)
cutoff_date = cumulative_share[cumulative_share <= 0.80].index.max()

development_raw = positive_source.loc[positive_source['sale_date'] <= cutoff_date].copy()
holdout_raw_locked = positive_source.loc[positive_source['sale_date'] > cutoff_date].copy()
invalid_target_quarantine = source.loc[source['price'] <= 0].copy()

split_report = pd.DataFrame({
    'partition': ['development', 'future_holdout', 'invalid_target_quarantine'],
    'rows': [len(development_raw), len(holdout_raw_locked), len(invalid_target_quarantine)],
    'date_min': [development_raw['sale_date'].min(), holdout_raw_locked['sale_date'].min(), invalid_target_quarantine['sale_date'].min()],
    'date_max': [development_raw['sale_date'].max(), holdout_raw_locked['sale_date'].max(), invalid_target_quarantine['sale_date'].max()],
})
split_report.to_csv(TABLE_DIR / '02_temporal_split_report.csv', index=False)
display(split_report)

assert len(development_raw) + len(holdout_raw_locked) + len(invalid_target_quarantine) == len(source)
assert development_raw['sale_date'].max() < holdout_raw_locked['sale_date'].min()

# Persist labels separately, then remove both the target and its direct derivative
# from the active holdout feature table.
LOCKED_HOLDOUT_LABEL_PATH = TABLE_DIR / '_LOCKED_future_holdout_labels.csv'
holdout_raw_locked[['source_row_id', 'price']].to_csv(LOCKED_HOLDOUT_LABEL_PATH, index=False)
holdout_source_ids = holdout_raw_locked['source_row_id'].to_numpy()
holdout_raw_features_only = holdout_raw_locked.drop(columns=['price', 'price_per_sqft']).copy()
del holdout_raw_locked

print('Cutoff date:', cutoff_date.date())
print('Locked labels:', LOCKED_HOLDOUT_LABEL_PATH)
print('Holdout targets and price_per_sqft are unavailable to development-stage code.')

## 2.3 Target distribution

Both the raw-dollar and log-price views are necessary. The log transformation is a modeling candidate because it reduces skewness and stabilizes proportional errors.

In [ ]:
train_price = development_raw['price']
target_summary = train_price.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame('price')
target_summary.loc['skewness'] = train_price.skew()
target_summary.loc['log1p_skewness'] = np.log1p(train_price).skew()
target_summary.to_csv(TABLE_DIR / '03_target_summary.csv')
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train_price, bins=50)
axes[0].axvline(train_price.median(), linestyle='--', label=f'Median ${train_price.median():,.0f}')
axes[0].set(title='Raw training-price distribution', xlabel='Price ($)', ylabel='Properties')
axes[0].legend()
axes[1].hist(np.log1p(train_price), bins=40)
axes[1].set(title='log(1 + price) distribution', xlabel='log(1 + price)', ylabel='Properties')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_target_raw_and_log.png', dpi=160, bbox_inches='tight')
plt.show()

## 2.4 Univariate feature distributions and categorical frequencies

In [ ]:
numeric_columns = [
    'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated',
]
numeric_summary = development_raw[numeric_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
numeric_summary['missing_n'] = development_raw[numeric_columns].isna().sum()
numeric_summary['zero_n'] = development_raw[numeric_columns].eq(0).sum()
numeric_summary['skewness'] = development_raw[numeric_columns].skew(numeric_only=True)
numeric_summary.to_csv(TABLE_DIR / '04_univariate_numeric_summary.csv')
display(numeric_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].hist(development_raw['sqft_living'], bins=40)
axes[0, 0].set(title='Living area', xlabel='Square feet')
axes[0, 1].hist(np.log1p(development_raw['sqft_lot']), bins=40)
axes[0, 1].set(title='Log lot area', xlabel='log(1 + sqft_lot)')
development_raw['condition'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set(title='Condition frequency', xlabel='Condition', ylabel='Count')
development_raw['city'].value_counts().head(15).sort_values().plot(kind='barh', ax=axes[1, 1])
axes[1, 1].set(title='Top 15 cities', xlabel='Count')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_univariate_feature_overview.png', dpi=160, bbox_inches='tight')
plt.show()

## 2.5 Bivariate and multivariate EDA

Pearson measures linear association, while Spearman measures monotonic rank association. Both are shown because the target contains extreme values.

In [ ]:
eda = development_raw.copy()
eda['property_age'] = eda['sale_date'].dt.year - eda['yr_built']
eda['has_basement'] = (eda['sqft_basement'] > 0).astype(int)
eda['log_sqft_lot'] = np.log1p(eda['sqft_lot'])
eda['log_price'] = np.log1p(eda['price'])

association_features = [
    'sqft_living', 'sqft_above', 'bathrooms', 'bedrooms', 'view',
    'sqft_basement', 'floors', 'waterfront', 'has_basement',
    'log_sqft_lot', 'yr_built', 'property_age', 'condition',
]
rows = []
for feature in association_features:
    rows.append({
        'feature': feature,
        'pearson_raw_price': eda[feature].corr(eda['price'], method='pearson'),
        'pearson_log_price': eda[feature].corr(eda['log_price'], method='pearson'),
        'spearman_price': eda[feature].corr(eda['price'], method='spearman'),
    })
association_table = pd.DataFrame(rows).sort_values('pearson_log_price', ascending=False)
association_table.to_csv(TABLE_DIR / '05_target_associations.csv', index=False)
display(association_table)

plot_data = association_table.sort_values('pearson_log_price')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_data['feature'], plot_data['pearson_log_price'])
ax.set(title='Pearson correlation with log(1 + price)', xlabel='Correlation')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_target_correlations.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Log-log living-area relationship and size-decile summary
x_log = np.log1p(eda[['sqft_living']])
y_log = eda['log_price']
size_model = LinearRegression().fit(x_log, y_log)
eda['fitted_log_price_size_only'] = size_model.predict(x_log)
eda['living_decile'] = pd.qcut(eda['sqft_living'], 10, labels=False, duplicates='drop') + 1
size_deciles = eda.groupby('living_decile').agg(
    median_sqft_living=('sqft_living', 'median'),
    median_price=('price', 'median'),
    price_q25=('price', lambda s: s.quantile(0.25)),
    price_q75=('price', lambda s: s.quantile(0.75)),
    n=('price', 'size'),
).reset_index()
size_deciles.to_csv(TABLE_DIR / '06_size_decile_summary.csv', index=False)
display(size_deciles)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(np.log1p(eda['sqft_living']), eda['log_price'], alpha=0.25, s=14)
order = np.argsort(np.log1p(eda['sqft_living']).to_numpy())
ax.plot(np.log1p(eda['sqft_living']).to_numpy()[order], eda['fitted_log_price_size_only'].to_numpy()[order])
ax.set(title='Log price versus log living area', xlabel='log(1 + sqft_living)', ylabel='log(1 + price)')
fig.tight_layout()
fig.savefig(FIG_DIR / '04_log_price_vs_log_living.png', dpi=160, bbox_inches='tight')
plt.show()

print('Estimated log-log slope:', float(size_model.coef_[0]))

In [ ]:
# Partial correlations after adjusting for log living area and city

def residualize(values: np.ndarray, controls: np.ndarray) -> np.ndarray:
    model = LinearRegression().fit(controls, values)
    return values - model.predict(controls)

city_dummies = pd.get_dummies(eda['city'], drop_first=True, dtype=float)
controls = np.column_stack([np.log1p(eda['sqft_living']).to_numpy(), city_dummies.to_numpy()])
price_residual = residualize(eda['log_price'].to_numpy(), controls)

partial_features = [
    'bedrooms', 'bathrooms', 'log_sqft_lot', 'floors', 'waterfront',
    'view', 'condition', 'sqft_basement', 'property_age', 'has_basement',
]
partial_rows = []
for feature in partial_features:
    feature_residual = residualize(eda[feature].to_numpy(dtype=float), controls)
    partial_rows.append({
        'feature': feature,
        'marginal_corr_log_price': eda[feature].corr(eda['log_price']),
        'partial_corr_given_log_living_and_city': np.corrcoef(feature_residual, price_residual)[0, 1],
    })
partial_table = pd.DataFrame(partial_rows).sort_values('partial_corr_given_log_living_and_city', ascending=False)
partial_table.to_csv(TABLE_DIR / '07_partial_correlations.csv', index=False)
display(partial_table)

In [ ]:
# Heteroscedasticity diagnostic: residual spread by fitted-value decile
raw_size = LinearRegression().fit(eda[['sqft_living']], eda['price'])
eda['raw_size_fitted'] = raw_size.predict(eda[['sqft_living']])
eda['raw_size_residual'] = eda['price'] - eda['raw_size_fitted']
eda['raw_fitted_decile'] = pd.qcut(eda['raw_size_fitted'], 10, labels=False, duplicates='drop') + 1

log_size = LinearRegression().fit(np.log1p(eda[['sqft_living']]), eda['log_price'])
eda['log_size_fitted'] = log_size.predict(np.log1p(eda[['sqft_living']]))
eda['log_size_residual'] = eda['log_price'] - eda['log_size_fitted']
eda['log_fitted_decile'] = pd.qcut(eda['log_size_fitted'], 10, labels=False, duplicates='drop') + 1

raw_spread = eda.groupby('raw_fitted_decile')['raw_size_residual'].agg(
    residual_iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
    median_abs_residual=lambda s: np.median(np.abs(s)),
).reset_index()
log_spread = eda.groupby('log_fitted_decile')['log_size_residual'].agg(
    residual_iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
    median_abs_residual=lambda s: np.median(np.abs(s)),
).reset_index()
raw_spread.to_csv(TABLE_DIR / '08_raw_residual_spread.csv', index=False)
log_spread.to_csv(TABLE_DIR / '09_log_residual_spread.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(raw_spread['raw_fitted_decile'], raw_spread['residual_iqr'], marker='o')
axes[0].set(title='Raw-price residual IQR', xlabel='Fitted-value decile', ylabel='Residual IQR ($)')
axes[1].plot(log_spread['log_fitted_decile'], log_spread['residual_iqr'], marker='o')
axes[1].set(title='Log-price residual IQR', xlabel='Fitted-value decile', ylabel='Residual IQR')
fig.tight_layout()
fig.savefig(FIG_DIR / '05_heteroscedasticity_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

# CRISP-DM Phase 3 — Data Preparation
## 3.1 Deterministic cleaning and validation

The cleaning strategy preserves raw values and creates validated companion fields. It does not guess corrections and does not delete statistical outliers.

In [ ]:
def normalize_display_text(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)


def canonicalize_source(df: pd.DataFrame, include_target: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = df.copy()
    if 'source_row_id' not in frame.columns:
        frame.insert(0, 'source_row_id', np.arange(1, len(frame) + 1, dtype=np.int64))

    date = pd.to_datetime(frame['date'], errors='coerce')
    statezip = normalize_display_text(frame['statezip']).str.extract(
        r'^([A-Za-z]{2})\s+(\d{5})(?:-\d{4})?$', expand=True
    )

    bedrooms = pd.to_numeric(frame['bedrooms'], errors='coerce')
    bathrooms = pd.to_numeric(frame['bathrooms'], errors='coerce')
    floors = pd.to_numeric(frame['floors'], errors='coerce')
    living = pd.to_numeric(frame['sqft_living'], errors='coerce')
    lot = pd.to_numeric(frame['sqft_lot'], errors='coerce')
    basement = pd.to_numeric(frame['sqft_basement'], errors='coerce')
    yr_built = pd.to_numeric(frame['yr_built'], errors='coerce')
    yr_renovated = pd.to_numeric(frame['yr_renovated'], errors='coerce').fillna(0)

    bedrooms_invalid = bedrooms.isna() | bedrooms.le(0) | ~np.isclose(bedrooms, np.round(bedrooms))
    bathrooms_invalid = bathrooms.isna() | bathrooms.le(0) | ~np.isclose(bathrooms * 4, np.round(bathrooms * 4))
    floors_invalid = floors.isna() | floors.le(0) | ~np.isclose(floors * 2, np.round(floors * 2))

    renovation_none = yr_renovated.eq(0)
    renovation_valid = yr_renovated.gt(0) & yr_renovated.ge(yr_built) & yr_renovated.le(date.dt.year)
    renovation_invalid = ~(renovation_none | renovation_valid)
    renovation_status = np.select(
        [renovation_none.to_numpy(), renovation_valid.to_numpy()],
        ['none_recorded', 'valid_year'], default='invalid_year'
    )

    cleaned = pd.DataFrame(index=frame.index)
    cleaned['meta_source_row_id'] = frame['source_row_id'].astype(int)
    cleaned['meta_sale_date'] = date.dt.strftime('%Y-%m-%d')
    cleaned['meta_address_group_id'] = (
        normalize_display_text(frame['street']).str.casefold() + '|' +
        normalize_display_text(frame['city']).str.casefold() + '|' +
        statezip[1].astype('string')
    ).map(lambda x: 'ADDR_' + hashlib.sha256(str(x).encode()).hexdigest()[:16])

    price = pd.to_numeric(frame['price'], errors='coerce') if 'price' in frame.columns else None
    if include_target:
        if price is None:
            raise ValueError('price is required when include_target=True')
        cleaned['price'] = price

    cleaned['bedrooms'] = bedrooms.mask(bedrooms_invalid)
    cleaned['bathrooms'] = bathrooms.mask(bathrooms_invalid)
    cleaned['sqft_living'] = living
    cleaned['sqft_lot'] = lot
    cleaned['floors'] = floors.mask(floors_invalid)
    cleaned['waterfront'] = pd.to_numeric(frame['waterfront'], errors='coerce')
    cleaned['view'] = pd.to_numeric(frame['view'], errors='coerce')
    cleaned['condition'] = pd.to_numeric(frame['condition'], errors='coerce')
    cleaned['property_age'] = date.dt.year - yr_built
    cleaned['has_basement'] = basement.gt(0).astype(int)
    cleaned['basement_share'] = basement / living
    cleaned['room_count_invalid'] = (bedrooms_invalid | bathrooms_invalid).astype(int)
    cleaned['years_since_renovation'] = np.where(renovation_valid, date.dt.year - yr_renovated, np.nan)
    cleaned['renovation_status'] = pd.Series(renovation_status, index=frame.index, dtype='string')
    cleaned['city'] = normalize_display_text(frame['city'])
    cleaned['zip_code'] = statezip[1].astype('string')

    flags = pd.DataFrame(index=frame.index)
    flags['date_parse_invalid'] = date.isna()
    flags['target_nonpositive_or_missing'] = (
        price.isna() | price.le(0) if price is not None
        else pd.Series(False, index=frame.index)
    )
    flags['bedrooms_invalid'] = bedrooms_invalid
    flags['bathrooms_invalid'] = bathrooms_invalid
    flags['floors_invalid'] = floors_invalid
    flags['renovation_year_invalid'] = renovation_invalid
    flags['statezip_parse_invalid'] = statezip.isna().any(axis=1)
    flags['area_identity_invalid'] = living.ne(pd.to_numeric(frame['sqft_above']) + basement)
    if price is not None and 'price_per_sqft' in frame.columns:
        flags['price_per_sqft_identity_invalid'] = ~np.isclose(
            pd.to_numeric(frame['price_per_sqft']), price / living, atol=0.011, rtol=0
        )
    else:
        flags['price_per_sqft_identity_invalid'] = False
    return cleaned.reset_index(drop=True), flags.reset_index(drop=True)


canonical_development, development_flags = canonicalize_source(development_raw, include_target=True)
canonical_holdout_features, holdout_feature_flags = canonicalize_source(holdout_raw_features_only, include_target=False)
canonical_quarantine, quarantine_flags = canonicalize_source(invalid_target_quarantine, include_target=True)

canonical_development['meta_sale_date'] = pd.to_datetime(canonical_development['meta_sale_date'])
canonical_holdout_features['meta_sale_date'] = pd.to_datetime(canonical_holdout_features['meta_sale_date'])
canonical_quarantine['meta_sale_date'] = pd.to_datetime(canonical_quarantine['meta_sale_date'])

quality_report = pd.DataFrame({
    'issue': development_flags.columns,
    'development_count': [int(development_flags[c].sum()) for c in development_flags.columns],
    'holdout_feature_count': [int(holdout_feature_flags[c].sum()) for c in holdout_feature_flags.columns],
    'quarantine_count': [int(quarantine_flags[c].sum()) for c in quarantine_flags.columns],
})
quality_report['available_total_count'] = quality_report[
    ['development_count', 'holdout_feature_count', 'quarantine_count']
].sum(axis=1)
quality_report.to_csv(TABLE_DIR / '10_data_quality_report.csv', index=False)
display(quality_report)

canonical_development.to_csv(TABLE_DIR / '11_canonical_development.csv', index=False)
canonical_holdout_features.to_csv(TABLE_DIR / '12_locked_holdout_features.csv', index=False)

assert canonical_development['price'].gt(0).all()
assert 'price' not in canonical_holdout_features.columns
assert 'price_per_sqft' not in canonical_holdout_features.columns
assert canonical_development['meta_sale_date'].max() < canonical_holdout_features['meta_sale_date'].min()
print('Canonical development shape:', canonical_development.shape)
print('Locked holdout feature shape:', canonical_holdout_features.shape)

# Remove full target-bearing source frames after canonical construction.
del source, raw_source, positive_source

## 3.2 Leakage-safe feature engineering and preprocessing

The preprocessing objects below remain **unfitted** until they are placed inside a temporal training fold.

In [ ]:
REQUIRED_CANONICAL_COLUMNS = [
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'property_age', 'has_basement',
    'basement_share', 'room_count_invalid', 'years_since_renovation',
    'renovation_status', 'city', 'zip_code',
]
PROHIBITED_INPUT_COLUMNS = {
    'price', 'price_per_sqft', 'price_per_sqft_recomputed', 'street',
    'address_key', 'source_row_id',
}


def make_one_hot_encoder(*, drop=None, min_frequency=None):
    kwargs = dict(handle_unknown='infrequent_if_exist', drop=drop, min_frequency=min_frequency)
    try:
        return OneHotEncoder(sparse_output=True, **kwargs)
    except TypeError:  # Compatibility with older sklearn.
        return OneHotEncoder(sparse=True, **kwargs)


class HousePriceFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y=None):
        self._validate(X)
        preview = self._engineer(X.iloc[: min(5, len(X))].copy())
        self.feature_names_out_ = np.asarray(preview.columns, dtype=object)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self._validate(X)
        return self._engineer(X.copy()).loc[:, self.feature_names_out_]

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_.copy()

    @staticmethod
    def _validate(X: pd.DataFrame):
        if not isinstance(X, pd.DataFrame):
            raise TypeError('A pandas DataFrame with named columns is required.')
        missing = [c for c in REQUIRED_CANONICAL_COLUMNS if c not in X.columns]
        if missing:
            raise ValueError(f'Missing canonical columns: {missing}')
        prohibited = sorted(PROHIBITED_INPUT_COLUMNS.intersection(X.columns))
        if prohibited:
            raise ValueError(f'Prohibited leakage/identifier fields found: {prohibited}')

    @staticmethod
    def _engineer(frame: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=frame.index)
        numeric = [
            'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
            'waterfront', 'view', 'condition', 'property_age', 'has_basement',
            'basement_share', 'room_count_invalid', 'years_since_renovation',
        ]
        for c in numeric:
            out[c] = pd.to_numeric(frame[c], errors='coerce')

        if (out['sqft_living'] <= 0).any() or (out['sqft_lot'] <= 0).any():
            raise ValueError('Living and lot areas must be positive.')
        if (~out['basement_share'].between(0, 1)).any():
            raise ValueError('basement_share must be in [0,1].')

        living = out['sqft_living']
        lot = out['sqft_lot']
        bedrooms = out['bedrooms'].where(out['bedrooms'] > 0)
        bathrooms = out['bathrooms'].where(out['bathrooms'] >= 0)
        share = out['basement_share']
        basement = (living * share).clip(lower=0)
        above = (living - basement).clip(lower=0)

        out['log_sqft_living'] = np.log1p(living)
        out['log_sqft_lot'] = np.log1p(lot)
        out['log_sqft_above'] = np.log1p(above)
        out['log_sqft_basement'] = np.log1p(basement)
        out['sqft_per_bedroom'] = living / bedrooms
        out['bathrooms_per_bedroom'] = bathrooms / bedrooms
        out['lot_to_living_ratio'] = lot / living
        out['log_lot_to_living_ratio'] = np.log1p(out['lot_to_living_ratio'])
        out['is_new_construction'] = (out['property_age'] == 0).astype(float)
        out['is_historic_100_plus'] = (out['property_age'] >= 100).astype(float)

        out['renovation_status'] = frame['renovation_status'].astype('string')
        out['city'] = frame['city'].astype('string')
        out['zip_code'] = frame['zip_code'].astype('string').str.zfill(5)
        out['floors_level'] = frame['floors'].astype('string')
        out['view_level'] = frame['view'].astype('string')
        out['condition_level'] = frame['condition'].astype('string')
        return out.replace([np.inf, -np.inf], np.nan)


def build_linear_preprocessor(location_mode='both', area_mode='composition', min_frequency=20, spline_knots=5):
    if area_mode == 'composition':
        area_continuous = ['basement_share']
        area_spline = ['log_sqft_living']
        area_binary = ['has_basement']
    elif area_mode == 'components':
        area_continuous = ['log_sqft_basement']
        area_spline = ['log_sqft_above']
        area_binary = ['has_basement']
    else:
        raise ValueError(area_mode)

    if location_mode == 'city':
        locations = ['city']
    elif location_mode == 'zip':
        locations = ['zip_code']
    elif location_mode == 'both':
        locations = ['city', 'zip_code']
    else:
        raise ValueError(location_mode)

    continuous = [
        'bedrooms', 'bathrooms', 'log_sqft_lot', 'sqft_per_bedroom',
        'bathrooms_per_bedroom', 'years_since_renovation',
        'log_lot_to_living_ratio', *area_continuous,
    ]
    splines = [*area_spline, 'property_age']
    binary = [
        'waterfront', 'room_count_invalid', 'is_new_construction',
        'is_historic_100_plus', *area_binary,
    ]
    discrete = ['floors_level', 'view_level', 'condition_level', 'renovation_status']

    continuous_pipeline = Pipeline([
        ('scaler', RobustScaler()),
        ('imputer', SimpleImputer(strategy='median')),
    ])
    spline_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('spline', SplineTransformer(
            n_knots=spline_knots, degree=3, knots='quantile',
            extrapolation='linear', include_bias=False,
        )),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', make_one_hot_encoder(drop='first', min_frequency=min_frequency)),
    ])
    location_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', make_one_hot_encoder(drop='first', min_frequency=min_frequency)),
    ])

    columns = ColumnTransformer([
        ('continuous', continuous_pipeline, continuous),
        ('spline', spline_pipeline, splines),
        ('binary', SimpleImputer(strategy='most_frequent'), binary),
        ('discrete', categorical_pipeline, discrete),
        ('location', location_pipeline, locations),
    ], remainder='drop', sparse_threshold=1.0, verbose_feature_names_out=True)

    return Pipeline([
        ('feature_engineering', HousePriceFeatureEngineer()),
        ('columns', columns),
    ])


def build_tree_preprocessor(location_mode='both', min_frequency=20):
    locations = {'city': ['city'], 'zip': ['zip_code'], 'both': ['city', 'zip_code']}[location_mode]
    numeric = [
        'bedrooms', 'bathrooms', 'sqft_living', 'log_sqft_living', 'log_sqft_lot',
        'sqft_per_bedroom', 'bathrooms_per_bedroom', 'log_lot_to_living_ratio',
        'property_age', 'basement_share', 'has_basement', 'waterfront', 'view',
        'condition', 'is_new_construction', 'is_historic_100_plus',
        'years_since_renovation', 'room_count_invalid',
    ]
    categorical = ['renovation_status', *locations]
    columns = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', make_one_hot_encoder(drop=None, min_frequency=min_frequency)),
        ]), categorical),
    ], sparse_threshold=1.0, verbose_feature_names_out=True)
    return Pipeline([
        ('feature_engineering', HousePriceFeatureEngineer()),
        ('columns', columns),
    ])

In [ ]:
# Re-establish output directories in case a prior interrupted run removed them.
for directory in (OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Feature-group mapping used for group-level feature selection and model restriction.
SELECTED_GROUPS = {
    'location', 'living_area', 'room_layout', 'age', 'lot_and_land_intensity',
    'view', 'basement', 'waterfront', 'condition',
}
FULL_GROUPS = SELECTED_GROUPS | {'renovation', 'floors', 'data_quality'}


def feature_group(name: str) -> str:
    if name.startswith('location__') or name.startswith('categorical__city') or name.startswith('categorical__zip_code'):
        return 'location'
    if 'log_sqft_living' in name or 'log_sqft_above' in name or name.endswith('__sqft_living'):
        return 'living_area'
    if 'log_sqft_lot' in name or 'lot_to_living' in name or name.endswith('__sqft_lot'):
        return 'lot_and_land_intensity'
    if any(token in name for token in ['bedrooms', 'bathrooms', 'sqft_per_bedroom', 'bathrooms_per_bedroom']):
        return 'room_layout'
    if 'basement_share' in name or 'has_basement' in name or 'log_sqft_basement' in name:
        return 'basement'
    if 'floors_level' in name:
        return 'floors'
    if 'waterfront' in name:
        return 'waterfront'
    if 'view_level' in name or name.endswith('__view'):
        return 'view'
    if 'condition_level' in name or name.endswith('__condition'):
        return 'condition'
    if 'property_age' in name or 'is_new_construction' in name or 'is_historic_100_plus' in name:
        return 'age'
    if 'years_since_renovation' in name or 'renovation_status' in name:
        return 'renovation'
    if 'room_count_invalid' in name:
        return 'data_quality'
    raise KeyError(f'Unmapped transformed feature: {name}')


def fit_transform_grouped(preprocessor, train_df, validation_df, groups):
    X_train_raw = train_df.drop(columns=['price'], errors='ignore')
    X_validation_raw = validation_df.drop(columns=['price'], errors='ignore')
    X_train = preprocessor.fit_transform(X_train_raw)
    X_validation = preprocessor.transform(X_validation_raw)
    names = np.asarray(preprocessor.get_feature_names_out(), dtype=str)
    mask = np.asarray([feature_group(name) in groups for name in names])
    return X_train[:, mask], X_validation[:, mask], names[mask]

# Reference-fit matrix audit on development data only.
reference_preprocessor = build_linear_preprocessor(location_mode='both')
X_reference_all = reference_preprocessor.fit_transform(canonical_development.drop(columns=['price']))
reference_names_all = np.asarray(reference_preprocessor.get_feature_names_out(), dtype=str)
reference_mask_9 = np.asarray([feature_group(n) in SELECTED_GROUPS for n in reference_names_all])
X_reference_9 = X_reference_all[:, reference_mask_9]

matrix_audit = pd.DataFrame([{
    'development_rows': X_reference_9.shape[0],
    'selected_transformed_features': X_reference_9.shape[1],
    'sparse_output': sparse.issparse(X_reference_9),
    'density': X_reference_9.nnz / np.prod(X_reference_9.shape) if sparse.issparse(X_reference_9) else np.count_nonzero(X_reference_9) / np.prod(X_reference_9.shape),
    'missing_values_after_transform': int(np.isnan(X_reference_9.data).sum()) if sparse.issparse(X_reference_9) else int(np.isnan(X_reference_9).sum()),
    'infinite_values_after_transform': int(np.isinf(X_reference_9.data).sum()) if sparse.issparse(X_reference_9) else int(np.isinf(X_reference_9).sum()),
}])
matrix_audit.to_csv(TABLE_DIR / '13_transformed_matrix_audit.csv', index=False)
display(matrix_audit)

## 3.3 Outlier, anomaly, leverage, and influence analysis

An outlier screen is a review mechanism, not an automatic deletion command.

In [ ]:
def tukey_summary(series: pd.Series, multiplier=1.5):
    s = pd.Series(series).dropna().astype(float)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - multiplier * iqr, q3 + multiplier * iqr
    return {
        'q1': q1, 'q3': q3, 'iqr': iqr, 'lower_fence': lower, 'upper_fence': upper,
        'low_flags': int((s < lower).sum()), 'high_flags': int((s > upper).sum()),
        'total_flags': int(((s < lower) | (s > upper)).sum()),
    }


def robust_mad_flags(series: pd.Series, threshold=3.5):
    s = pd.Series(series, dtype=float)
    median = s.median()
    mad = np.median(np.abs(s.dropna() - median))
    if mad == 0 or not np.isfinite(mad):
        return pd.Series(False, index=s.index), np.nan
    z = 0.6745 * (s - median) / mad
    return z.abs() > threshold, mad

outlier_variables = {
    'price_raw': canonical_development['price'],
    'price_log1p': np.log1p(canonical_development['price']),
    'sqft_living_raw': canonical_development['sqft_living'],
    'sqft_living_log1p': np.log1p(canonical_development['sqft_living']),
    'sqft_lot_raw': canonical_development['sqft_lot'],
    'sqft_lot_log1p': np.log1p(canonical_development['sqft_lot']),
    'bedrooms': canonical_development['bedrooms'],
    'bathrooms': canonical_development['bathrooms'],
    'property_age': canonical_development['property_age'],
}
outlier_rows = []
for name, values in outlier_variables.items():
    t15 = tukey_summary(values, 1.5)
    t30 = tukey_summary(values, 3.0)
    mad_flags, mad = robust_mad_flags(values)
    outlier_rows.append({
        'variable': name,
        'tukey_1_5_flags': t15['total_flags'],
        'tukey_3_0_flags': t30['total_flags'],
        'mad_3_5_flags': int(mad_flags.sum()),
        'tukey_1_5_lower': t15['lower_fence'],
        'tukey_1_5_upper': t15['upper_fence'],
    })
outlier_table = pd.DataFrame(outlier_rows)
outlier_table.to_csv(TABLE_DIR / '14_univariate_outlier_screens.csv', index=False)
display(outlier_table)

In [ ]:
# Predictor-only multivariate anomaly analysis

def build_structural_features(canonical: pd.DataFrame) -> pd.DataFrame:
    x = pd.DataFrame(index=canonical.index)
    x['log_sqft_living'] = np.log1p(canonical['sqft_living'].astype(float))
    x['log_sqft_lot'] = np.log1p(canonical['sqft_lot'].astype(float))
    x['bedrooms'] = canonical['bedrooms'].astype(float)
    x['bathrooms'] = canonical['bathrooms'].astype(float)
    x['property_age'] = canonical['property_age'].astype(float)
    x['basement_share'] = canonical['basement_share'].astype(float)
    x['floors_scaled'] = canonical['floors'].astype(float) / 3.5
    x['view_scaled'] = canonical['view'].astype(float) / 4.0
    x['condition_scaled'] = (canonical['condition'].astype(float) - 1.0) / 4.0
    x['waterfront'] = canonical['waterfront'].astype(float)
    x['has_basement'] = canonical['has_basement'].astype(float)
    return x

structural_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler(quantile_range=(10, 90))),
])
X_struct = structural_preprocessor.fit_transform(build_structural_features(canonical_development))
X_struct_holdout = structural_preprocessor.transform(build_structural_features(canonical_holdout_features))

iforest = IsolationForest(
    n_estimators=N_IFOREST_TREES,
    max_samples=min(1024, len(canonical_development)),
    contamination='auto', random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_struct)
if_score_dev = iforest.decision_function(X_struct)
if_threshold = np.quantile(if_score_dev, 0.01)

mcd = MinCovDet(support_fraction=0.75, random_state=RANDOM_STATE).fit(X_struct)
md2_dev = mcd.mahalanobis(X_struct)
md_threshold = np.quantile(md2_dev, 0.99)

anomaly_scores = canonical_development[['meta_source_row_id', 'meta_sale_date', 'price']].copy()
anomaly_scores['isolation_score'] = if_score_dev
anomaly_scores['robust_mahalanobis_sq'] = md2_dev
anomaly_scores['isolation_top_1pct'] = if_score_dev <= if_threshold
anomaly_scores['robust_distance_top_1pct'] = md2_dev >= md_threshold
anomaly_scores['consensus_anomaly'] = anomaly_scores['isolation_top_1pct'] & anomaly_scores['robust_distance_top_1pct']
anomaly_scores.to_csv(TABLE_DIR / '15_structural_anomaly_scores.csv', index=False)

anomaly_summary = pd.DataFrame({
    'screen': ['Isolation Forest', 'Robust distance', 'Consensus'],
    'development_flags': [
        int(anomaly_scores['isolation_top_1pct'].sum()),
        int(anomaly_scores['robust_distance_top_1pct'].sum()),
        int(anomaly_scores['consensus_anomaly'].sum()),
    ],
})
display(anomaly_summary)

In [ ]:
# Leverage and Cook's distance on the log-target reference linear design.
# This is a diagnostic model, not the final predictive estimator.
X_inf = X_reference_9.toarray() if sparse.issparse(X_reference_9) else np.asarray(X_reference_9)
X_inf = np.column_stack([np.ones(len(X_inf)), X_inf])
y_inf = np.log1p(canonical_development['price'].to_numpy(float))

xtx_inv = np.linalg.pinv(X_inf.T @ X_inf)
beta = xtx_inv @ X_inf.T @ y_inf
fitted = X_inf @ beta
residual = y_inf - fitted
n_inf, p_inf = X_inf.shape
hat_diag = np.einsum('ij,jk,ik->i', X_inf, xtx_inv, X_inf)
sigma2 = np.sum(residual ** 2) / max(n_inf - p_inf, 1)
studentized = residual / np.sqrt(np.maximum(sigma2 * (1 - hat_diag), 1e-12))
cooks = (studentized ** 2 / p_inf) * (hat_diag / np.maximum(1 - hat_diag, 1e-12))

influence = canonical_development[['meta_source_row_id', 'meta_sale_date', 'price']].copy()
influence['leverage'] = hat_diag
influence['studentized_residual'] = studentized
influence['cooks_distance'] = cooks
influence['high_leverage_2p_n'] = hat_diag > (2 * p_inf / n_inf)
influence['cooks_over_4_n'] = cooks > (4 / n_inf)
influence.to_csv(TABLE_DIR / '16_influence_diagnostics.csv', index=False)

display(pd.DataFrame({
    'metric': ['parameters_including_intercept', 'average_leverage', 'high_leverage_2p_n', 'cooks_over_4_n'],
    'value': [p_inf, p_inf / n_inf, int(influence['high_leverage_2p_n'].sum()), int(influence['cooks_over_4_n'].sum())],
}))

## 3.4 Multi-method feature selection

Selection is conducted at the **conceptual feature-group level**, not by arbitrarily dropping one spline basis or one ZIP indicator.

In [ ]:
# Filter evidence: mutual information and absolute Spearman correlation.
y_feature_selection = np.log1p(canonical_development['price'].to_numpy(float))
X_fs = X_reference_all.toarray() if sparse.issparse(X_reference_all) else np.asarray(X_reference_all)

# Treat every transformed column numerically for a consistent screening comparison.
mi = mutual_info_regression(X_fs, y_feature_selection, random_state=RANDOM_STATE)
spearman = np.array([
    pd.Series(X_fs[:, j]).corr(pd.Series(y_feature_selection), method='spearman')
    for j in range(X_fs.shape[1])
])
groups_all = np.asarray([feature_group(n) for n in reference_names_all])

filter_rows = []
for group in sorted(set(groups_all)):
    mask = groups_all == group
    mi_group = np.sort(mi[mask])[::-1]
    sp_group = np.sort(np.abs(spearman[mask]))[::-1]
    filter_rows.append({
        'group': group,
        'feature_columns': int(mask.sum()),
        'max_mutual_information': float(mi_group[0]),
        'top3_mean_mutual_information': float(mi_group[: min(3, len(mi_group))].mean()),
        'max_abs_spearman': float(sp_group[0]),
        'top3_mean_abs_spearman': float(sp_group[: min(3, len(sp_group))].mean()),
    })
filter_evidence = pd.DataFrame(filter_rows)
filter_evidence['filter_rank'] = (
    filter_evidence['top3_mean_mutual_information'].rank(ascending=False) +
    filter_evidence['top3_mean_abs_spearman'].rank(ascending=False)
) / 2
filter_evidence = filter_evidence.sort_values('filter_rank')
filter_evidence.to_csv(TABLE_DIR / '17_filter_feature_group_evidence.csv', index=False)
display(filter_evidence)

In [ ]:
# Embedded evidence: Elastic Net group stability across the three temporal folds.
@dataclass(frozen=True)
class TemporalFold:
    name: str
    train_end: pd.Timestamp
    validation_start: pd.Timestamp
    validation_end: pd.Timestamp

OUTER_FOLDS = [
    TemporalFold('fold_1', pd.Timestamp('2014-05-17'), pd.Timestamp('2014-05-18'), pd.Timestamp('2014-05-30')),
    TemporalFold('fold_2', pd.Timestamp('2014-05-30'), pd.Timestamp('2014-05-31'), pd.Timestamp('2014-06-12')),
    TemporalFold('fold_3', pd.Timestamp('2014-06-12'), pd.Timestamp('2014-06-13'), pd.Timestamp('2014-06-25')),
]


def fold_data(df: pd.DataFrame, fold: TemporalFold):
    train = df.loc[df['meta_sale_date'] <= fold.train_end].copy()
    validation = df.loc[df['meta_sale_date'].between(fold.validation_start, fold.validation_end)].copy()
    assert train['meta_sale_date'].max() < validation['meta_sale_date'].min()
    return train, validation

embedded_rows = []
embedded_alphas = {'fold_1': 0.003, 'fold_2': 0.0003, 'fold_3': 0.0003}
ACTIVE_FOLDS = OUTER_FOLDS[:1] if SMOKE_MODE else OUTER_FOLDS

for fold in ACTIVE_FOLDS:
    tr, va = fold_data(canonical_development, fold)
    pre = build_linear_preprocessor(location_mode='both')
    Xtr = pre.fit_transform(tr.drop(columns=['price']))
    names = np.asarray(pre.get_feature_names_out(), dtype=str)
    model = ElasticNet(
        alpha=embedded_alphas[fold.name], l1_ratio=0.80,
        max_iter=30000, tol=1e-5, random_state=RANDOM_STATE,
    ).fit(Xtr, np.log1p(tr['price']))
    coefficients = np.asarray(model.coef_)
    feature_groups = np.asarray([feature_group(n) for n in names])
    for group in sorted(set(feature_groups)):
        mask = feature_groups == group
        embedded_rows.append({
            'fold': fold.name,
            'group': group,
            'selected': bool(np.any(np.abs(coefficients[mask]) > 1e-10)),
            'l1_coefficient_mass': float(np.abs(coefficients[mask]).sum()),
        })
embedded_detail = pd.DataFrame(embedded_rows)
embedded_stability = embedded_detail.groupby('group').agg(
    folds_selected=('selected', 'sum'),
    selection_frequency=('selected', 'mean'),
    mean_l1_coefficient_mass=('l1_coefficient_mass', 'mean'),
).reset_index().sort_values(['folds_selected', 'mean_l1_coefficient_mass'], ascending=False)
embedded_stability.to_csv(TABLE_DIR / '18_embedded_feature_group_stability.csv', index=False)
display(embedded_stability)

In [ ]:
# The consensus set carried into modeling.
selection_manifest = pd.DataFrame({
    'feature_group': sorted(FULL_GROUPS),
    'selected_primary_9': [g in SELECTED_GROUPS for g in sorted(FULL_GROUPS)],
    'role': [
        'primary predictor group' if g in SELECTED_GROUPS else 'full-model sensitivity group'
        for g in sorted(FULL_GROUPS)
    ],
})
selection_manifest.to_csv(TABLE_DIR / '19_selected_feature_groups.csv', index=False)
display(selection_manifest)

# CRISP-DM Phase 4 — Modeling
## 4.1 Target-free clustering and market segmentation

Price is excluded while forming clusters. Price is overlaid only after the cluster count and assignments are fixed.

In [ ]:
cluster_frame = build_structural_features(canonical_development)
cluster_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler(quantile_range=(10, 90))),
])
X_cluster = cluster_preprocessor.fit_transform(cluster_frame)

cluster_evaluation_rows = []
cluster_label_store = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=15 if FAST_MODE else 25, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_cluster)
    cluster_label_store[k] = labels
    min_share = pd.Series(labels).value_counts(normalize=True).min()
    sil_sample = min(len(X_cluster), 2000 if FAST_MODE else len(X_cluster))
    silhouette = silhouette_score(X_cluster, labels, sample_size=sil_sample, random_state=RANDOM_STATE)

    resampled_partitions = []
    rng = np.random.default_rng(RANDOM_STATE + k)
    for _ in range(N_CLUSTER_RESAMPLES):
        sample_idx = rng.choice(len(X_cluster), size=int(0.80 * len(X_cluster)), replace=False)
        km_r = KMeans(n_clusters=k, n_init=10, random_state=int(rng.integers(0, 10_000)))
        km_r.fit(X_cluster[sample_idx])
        resampled_partitions.append(km_r.predict(X_cluster))
    aris = [
        adjusted_rand_score(labels, partition)
        for partition in resampled_partitions
    ]
    cluster_evaluation_rows.append({
        'k': k,
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin_score(X_cluster, labels),
        'calinski_harabasz': calinski_harabasz_score(X_cluster, labels),
        'minimum_cluster_share': min_share,
        'mean_resampling_ari': np.mean(aris),
    })
cluster_evaluation = pd.DataFrame(cluster_evaluation_rows)
for metric, ascending in [
    ('silhouette', False), ('davies_bouldin', True), ('calinski_harabasz', False),
    ('minimum_cluster_share', False), ('mean_resampling_ari', False),
]:
    cluster_evaluation[f'{metric}_rank'] = cluster_evaluation[metric].rank(ascending=ascending)
cluster_evaluation['composite_rank'] = cluster_evaluation.filter(like='_rank').mean(axis=1)
cluster_evaluation.to_csv(TABLE_DIR / '20_cluster_count_evaluation.csv', index=False)
display(cluster_evaluation.sort_values('k'))

SELECTED_K = 3  # Chosen for separation, stability, balance, and interpretability.
cluster_model = KMeans(n_clusters=SELECTED_K, n_init=25, random_state=RANDOM_STATE).fit(X_cluster)
cluster_labels = cluster_model.labels_

In [ ]:
cluster_profiles = canonical_development.assign(cluster_id=cluster_labels).groupby('cluster_id').agg(
    n=('price', 'size'),
    share=('price', lambda s: len(s) / len(canonical_development)),
    median_sqft_living=('sqft_living', 'median'),
    median_sqft_lot=('sqft_lot', 'median'),
    median_bedrooms=('bedrooms', 'median'),
    median_bathrooms=('bathrooms', 'median'),
    median_floors=('floors', 'median'),
    median_property_age=('property_age', 'median'),
    basement_rate=('has_basement', 'mean'),
    median_basement_share=('basement_share', 'median'),
    median_condition=('condition', 'median'),
    waterfront_rate=('waterfront', 'mean'),
).reset_index()

basement_cluster = int(cluster_profiles.loc[cluster_profiles['basement_rate'].idxmax(), 'cluster_id'])
remaining_profiles = cluster_profiles.loc[cluster_profiles['cluster_id'] != basement_cluster]
compact_cluster = int(remaining_profiles.loc[remaining_profiles['median_sqft_living'].idxmin(), 'cluster_id'])
newer_cluster = int([c for c in cluster_profiles['cluster_id'] if c not in {basement_cluster, compact_cluster}][0])
cluster_name_map = {
    compact_cluster: 'Compact older non-basement',
    basement_cluster: 'Older basement-oriented',
    newer_cluster: 'Newer larger multi-story',
}
cluster_profiles['cluster_name'] = cluster_profiles['cluster_id'].map(cluster_name_map)
cluster_profiles.to_csv(TABLE_DIR / '21_target_free_cluster_profiles.csv', index=False)
display(cluster_profiles.sort_values('cluster_id'))

# Price overlay after clusters are fixed.
price_overlay = canonical_development.assign(cluster_id=cluster_labels).groupby('cluster_id')['price'].agg(
    n='size', median_price='median', price_q25=lambda s: s.quantile(0.25),
    price_q75=lambda s: s.quantile(0.75), price_q90=lambda s: s.quantile(0.90),
).reset_index()
price_overlay['cluster_name'] = price_overlay['cluster_id'].map(cluster_name_map)
price_overlay.to_csv(TABLE_DIR / '22_posthoc_cluster_price_overlay.csv', index=False)
display(price_overlay)

pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_cluster)
coords = pca.transform(X_cluster)
fig, ax = plt.subplots(figsize=(9, 6))
for cluster_id, cluster_name in cluster_name_map.items():
    mask = cluster_labels == cluster_id
    ax.scatter(coords[mask, 0], coords[mask, 1], s=12, alpha=0.45, label=cluster_name)
ax.set(title=f'PCA projection of target-free K={SELECTED_K} clusters', xlabel='PC1', ylabel='PC2')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / '06_cluster_pca_projection.png', dpi=160, bbox_inches='tight')
plt.show()
print('PCA variance explained:', pca.explained_variance_ratio_)

## 4.2 Temporal regression benchmark

The benchmark includes dummy baselines, size-only regression, OLS, Ridge, Lasso, Elastic Net, Huber, Random Forest, Extra Trees, Gradient Boosting, and Histogram Gradient Boosting.

In [ ]:
def regression_metrics(y_true, prediction):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(prediction, dtype=float)
    p_nonnegative = np.clip(p, 0, None)
    return {
        'mae': mean_absolute_error(y, p),
        'rmse': mean_squared_error(y, p) ** 0.5,
        'median_ae': median_absolute_error(y, p),
        'r2': r2_score(y, p),
        'rmsle': np.sqrt(np.mean((np.log1p(y) - np.log1p(p_nonnegative)) ** 2)),
        'wape': np.abs(y - p).sum() / y.sum(),
        'mean_error': np.mean(p - y),
        'negative_prediction_n': int((p < 0).sum()),
    }


def fit_predict_estimator(estimator, X_train, y_train, X_validation, target_scale='log'):
    y_model = np.log1p(y_train) if target_scale == 'log' else np.asarray(y_train, dtype=float)
    estimator.fit(X_train, y_model)
    prediction_model_scale = estimator.predict(X_validation)
    prediction = np.expm1(prediction_model_scale) if target_scale == 'log' else prediction_model_scale
    return np.asarray(prediction, dtype=float)


def dense_if_needed(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)

# Fixed low-compute hyperparameters selected from earlier development-only temporal searches.
LINEAR_ALPHA_BY_FOLD = {
    'fold_1': {'ridge': 10.0, 'lasso': 0.0030, 'elastic': 0.0030, 'ridge_full': 1.0},
    'fold_2': {'ridge': 1.0, 'lasso': 0.0003, 'elastic': 0.0003, 'ridge_full': 1.0},
    'fold_3': {'ridge': 1.0, 'lasso': 0.0003, 'elastic': 0.0003, 'ridge_full': 1.0},
}

all_fold_metrics = []
all_oof_predictions = []

for fold in ACTIVE_FOLDS:
    train_fold, validation_fold = fold_data(canonical_development, fold)
    y_train = train_fold['price'].to_numpy(float)
    y_validation = validation_fold['price'].to_numpy(float)

    # Baselines
    baseline_specs = [
        ('dummy_median', DummyRegressor(strategy='median')),
        ('dummy_mean', DummyRegressor(strategy='mean')),
    ]
    dummy_X_train = np.zeros((len(train_fold), 1))
    dummy_X_validation = np.zeros((len(validation_fold), 1))
    for model_name, estimator in baseline_specs:
        prediction = fit_predict_estimator(estimator, dummy_X_train, y_train, dummy_X_validation, target_scale='raw')
        all_fold_metrics.append({'fold': fold.name, 'model': model_name, **regression_metrics(y_validation, prediction)})
        all_oof_predictions.extend([
            {'fold': fold.name, 'model': model_name, 'meta_source_row_id': row_id, 'actual': actual, 'prediction': pred}
            for row_id, actual, pred in zip(validation_fold['meta_source_row_id'], y_validation, prediction)
        ])

    # Size-only baselines
    for model_name, target_scale, transform_x in [
        ('size_only_raw', 'raw', False), ('size_only_log', 'log', True),
    ]:
        Xtr_size = train_fold[['sqft_living']].to_numpy(float)
        Xva_size = validation_fold[['sqft_living']].to_numpy(float)
        if transform_x:
            Xtr_size = np.log1p(Xtr_size)
            Xva_size = np.log1p(Xva_size)
        prediction = fit_predict_estimator(LinearRegression(), Xtr_size, y_train, Xva_size, target_scale)
        all_fold_metrics.append({'fold': fold.name, 'model': model_name, **regression_metrics(y_validation, prediction)})
        all_oof_predictions.extend([
            {'fold': fold.name, 'model': model_name, 'meta_source_row_id': row_id, 'actual': actual, 'prediction': pred}
            for row_id, actual, pred in zip(validation_fold['meta_source_row_id'], y_validation, prediction)
        ])

    # OLS with full-rank ZIP-only representation.
    pre_zip = build_linear_preprocessor(location_mode='zip')
    Xtr_zip, Xva_zip, _ = fit_transform_grouped(pre_zip, train_fold, validation_fold, SELECTED_GROUPS)
    for model_name, target_scale in [('ols_raw_zip_9', 'raw'), ('ols_log_zip_9', 'log')]:
        prediction = fit_predict_estimator(LinearRegression(), Xtr_zip, y_train, Xva_zip, target_scale)
        all_fold_metrics.append({'fold': fold.name, 'model': model_name, **regression_metrics(y_validation, prediction)})
        all_oof_predictions.extend([
            {'fold': fold.name, 'model': model_name, 'meta_source_row_id': row_id, 'actual': actual, 'prediction': pred}
            for row_id, actual, pred in zip(validation_fold['meta_source_row_id'], y_validation, prediction)
        ])

    # Combined city+ZIP regularized and robust models.
    pre_both = build_linear_preprocessor(location_mode='both')
    Xtr_both_all = pre_both.fit_transform(train_fold.drop(columns=['price']))
    Xva_both_all = pre_both.transform(validation_fold.drop(columns=['price']))
    names_both = np.asarray(pre_both.get_feature_names_out(), dtype=str)
    groups_both = np.asarray([feature_group(n) for n in names_both])
    mask9 = np.isin(groups_both, sorted(SELECTED_GROUPS))
    mask12 = np.isin(groups_both, sorted(FULL_GROUPS))
    Xtr9, Xva9 = Xtr_both_all[:, mask9], Xva_both_all[:, mask9]
    Xtr12, Xva12 = Xtr_both_all[:, mask12], Xva_both_all[:, mask12]
    alpha = LINEAR_ALPHA_BY_FOLD[fold.name]

    linear_specs = [
        ('ridge_log_both_9', Ridge(alpha=alpha['ridge'], solver='lsqr'), 'log', Xtr9, Xva9),
        ('ridge_log_both_full12', Ridge(alpha=alpha['ridge_full'], solver='lsqr'), 'log', Xtr12, Xva12),
        ('lasso_log_both_9', Lasso(alpha=alpha['lasso'], max_iter=30000, tol=1e-5), 'log', Xtr9, Xva9),
        ('elastic_log_both_9', ElasticNet(alpha=alpha['elastic'], l1_ratio=0.80, max_iter=30000, tol=1e-5, random_state=RANDOM_STATE), 'log', Xtr9, Xva9),
        ('huber_log_both_9', HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000, tol=1e-5), 'log', Xtr9, Xva9),
    ]
    if not FAST_MODE:
        linear_specs.extend([
            ('ridge_raw_both_9', Ridge(alpha=alpha['ridge'], solver='lsqr'), 'raw', Xtr9, Xva9),
            ('huber_raw_both_9', HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000, tol=1e-5), 'raw', Xtr9, Xva9),
        ])
    for model_name, estimator, target_scale, Xtr_model, Xva_model in linear_specs:
        prediction = fit_predict_estimator(estimator, Xtr_model, y_train, Xva_model, target_scale)
        all_fold_metrics.append({'fold': fold.name, 'model': model_name, **regression_metrics(y_validation, prediction)})
        all_oof_predictions.extend([
            {'fold': fold.name, 'model': model_name, 'meta_source_row_id': row_id, 'actual': actual, 'prediction': pred}
            for row_id, actual, pred in zip(validation_fold['meta_source_row_id'], y_validation, prediction)
        ])

    # Tree matrices and nonlinear models.
    tree_pre = build_tree_preprocessor(location_mode='both')
    Xtr_tree_all = tree_pre.fit_transform(train_fold.drop(columns=['price']))
    Xva_tree_all = tree_pre.transform(validation_fold.drop(columns=['price']))
    tree_names = np.asarray(tree_pre.get_feature_names_out(), dtype=str)
    tree_groups = np.asarray([feature_group(n) for n in tree_names])
    tree_mask = np.isin(tree_groups, sorted(SELECTED_GROUPS))
    Xtr_tree = Xtr_tree_all[:, tree_mask]
    Xva_tree = Xva_tree_all[:, tree_mask]

    tree_specs = [
        ('random_forest_log_9', RandomForestRegressor(n_estimators=N_TREES, max_features=0.7, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1), 'log', False),
        ('extra_trees_log_9', ExtraTreesRegressor(n_estimators=N_TREES, max_features=0.7, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1), 'log', False),
        ('gradient_boosting_log_9', GradientBoostingRegressor(n_estimators=20 if SMOKE_MODE else (55 if FAST_MODE else 110), learning_rate=0.045, max_depth=2, min_samples_leaf=10, loss='huber', random_state=RANDOM_STATE), 'log', True),
        ('hist_gradient_boosting_log_9', HistGradientBoostingRegressor(max_iter=25 if SMOKE_MODE else (80 if FAST_MODE else 140), learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, loss='squared_error', random_state=RANDOM_STATE), 'log', True),
    ]
    if not FAST_MODE:
        tree_specs.extend([
            ('random_forest_raw_9', RandomForestRegressor(n_estimators=N_TREES, max_features=0.7, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1), 'raw', False),
            ('extra_trees_raw_9', ExtraTreesRegressor(n_estimators=N_TREES, max_features=0.7, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1), 'raw', False),
            ('gradient_boosting_raw_9', GradientBoostingRegressor(n_estimators=110, learning_rate=0.045, max_depth=2, min_samples_leaf=10, loss='huber', random_state=RANDOM_STATE), 'raw', True),
            ('hist_gradient_boosting_raw_9', HistGradientBoostingRegressor(max_iter=140, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, loss='squared_error', random_state=RANDOM_STATE), 'raw', True),
        ])
    for model_name, estimator, target_scale, needs_dense in tree_specs:
        Xtr_model = dense_if_needed(Xtr_tree) if needs_dense else Xtr_tree
        Xva_model = dense_if_needed(Xva_tree) if needs_dense else Xva_tree
        prediction = fit_predict_estimator(estimator, Xtr_model, y_train, Xva_model, target_scale)
        all_fold_metrics.append({'fold': fold.name, 'model': model_name, **regression_metrics(y_validation, prediction)})
        all_oof_predictions.extend([
            {'fold': fold.name, 'model': model_name, 'meta_source_row_id': row_id, 'actual': actual, 'prediction': pred}
            for row_id, actual, pred in zip(validation_fold['meta_source_row_id'], y_validation, prediction)
        ])

fold_metrics = pd.DataFrame(all_fold_metrics)
oof_predictions = pd.DataFrame(all_oof_predictions)
fold_metrics.to_csv(TABLE_DIR / '23_temporal_fold_model_metrics.csv', index=False)
oof_predictions.to_csv(TABLE_DIR / '24_temporal_oof_predictions.csv', index=False)
print('Completed temporal model benchmark.')

In [ ]:
# Aggregate all out-of-fold predictions so metrics are calculated on the same 2,694 observations.
aggregate_rows = []
for model_name, group in oof_predictions.groupby('model'):
    aggregate_rows.append({'model': model_name, 'n': len(group), **regression_metrics(group['actual'], group['prediction'])})
aggregate_metrics = pd.DataFrame(aggregate_rows).sort_values('mae').reset_index(drop=True)
aggregate_metrics.insert(0, 'mae_rank', np.arange(1, len(aggregate_metrics) + 1))
aggregate_metrics.to_csv(TABLE_DIR / '25_aggregate_model_metrics.csv', index=False)
display(aggregate_metrics)

fig, ax = plt.subplots(figsize=(10, 8))
plot_metrics = aggregate_metrics.sort_values('mae', ascending=True)
ax.barh(plot_metrics['model'], plot_metrics['mae'])
ax.set(title='Temporal out-of-fold MAE by model', xlabel='MAE ($)')
fig.tight_layout()
fig.savefig(FIG_DIR / '07_model_mae_ranking.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Price-segment diagnostics based on each fold's training thresholds.
segment_rows = []
for fold in ACTIVE_FOLDS:
    train_fold, validation_fold = fold_data(canonical_development, fold)
    p90, p99 = train_fold['price'].quantile([0.90, 0.99])
    validation_ids = set(validation_fold['meta_source_row_id'])
    fold_oof = oof_predictions.loc[oof_predictions['fold'].eq(fold.name)].copy()
    fold_oof['segment'] = np.select(
        [fold_oof['actual'] > p99, fold_oof['actual'] > p90],
        ['above_training_p99', 'above_training_p90'],
        default='at_or_below_training_p90',
    )
    for (model_name, segment), group in fold_oof.groupby(['model', 'segment']):
        segment_rows.append({'fold': fold.name, 'model': model_name, 'segment': segment, **regression_metrics(group['actual'], group['prediction'])})
segment_metrics = pd.DataFrame(segment_rows)
segment_metrics.to_csv(TABLE_DIR / '26_price_segment_metrics.csv', index=False)

key_models = ['huber_log_both_9', 'ridge_log_both_9', 'hist_gradient_boosting_log_9']
segment_summary = segment_metrics.loc[segment_metrics['model'].isin(key_models)].groupby(['model', 'segment']).agg(
    n=('mae', 'count'), mean_fold_mae=('mae', 'mean'), mean_fold_rmse=('rmse', 'mean'), mean_error=('mean_error', 'mean')
).reset_index()
display(segment_summary)

# CRISP-DM Phase 5 — Final Evaluation
## 5.1 Freeze the finalists before opening holdout labels

The primary architecture is frozen from development evidence. Holdout predictions are created before holdout prices are accessed by the evaluation code.

In [ ]:
# Frozen finalists based on development results.
FROZEN_PRIMARY_MODEL = 'huber_log_both_9'
FROZEN_CHALLENGERS = ['ridge_log_both_9', 'ridge_log_both_full12', 'hist_gradient_boosting_log_9']

freeze_manifest = {
    'frozen_before_holdout_evaluation': True,
    'primary_model': FROZEN_PRIMARY_MODEL,
    'selected_groups': sorted(SELECTED_GROUPS),
    'huber_parameters': {'epsilon': 1.35, 'alpha': 0.0001, 'max_iter': 2000, 'tol': 1e-5},
    'location_mode': 'city_plus_zip',
    'area_mode': 'composition',
    'min_location_frequency': 20,
    'spline_knots': 5,
    'challengers': FROZEN_CHALLENGERS,
}
with open(TABLE_DIR / '27_frozen_model_decision.json', 'w') as handle:
    json.dump(freeze_manifest, handle, indent=2)
print(json.dumps(freeze_manifest, indent=2))

In [ ]:
# Fit the frozen primary and preregistered challengers on all development data.
X_dev_raw = canonical_development.drop(columns=['price'])
X_hold_raw = canonical_holdout_features.copy()
y_dev = canonical_development['price'].to_numpy(float)

# Linear preprocessor and selected/full matrices.
final_linear_pre = build_linear_preprocessor(location_mode='both')
X_dev_linear_all = final_linear_pre.fit_transform(X_dev_raw)
X_hold_linear_all = final_linear_pre.transform(X_hold_raw)
final_linear_names = np.asarray(final_linear_pre.get_feature_names_out(), dtype=str)
final_linear_groups = np.asarray([feature_group(n) for n in final_linear_names])
final_mask9 = np.isin(final_linear_groups, sorted(SELECTED_GROUPS))
final_mask12 = np.isin(final_linear_groups, sorted(FULL_GROUPS))

huber_final = HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000, tol=1e-5)
huber_final.fit(X_dev_linear_all[:, final_mask9], np.log1p(y_dev))
pred_huber = np.expm1(huber_final.predict(X_hold_linear_all[:, final_mask9]))

ridge9_final = Ridge(alpha=1.0, solver='lsqr').fit(X_dev_linear_all[:, final_mask9], np.log1p(y_dev))
pred_ridge9 = np.expm1(ridge9_final.predict(X_hold_linear_all[:, final_mask9]))

ridge12_final = Ridge(alpha=0.1, solver='lsqr').fit(X_dev_linear_all[:, final_mask12], np.log1p(y_dev))
pred_ridge12 = np.expm1(ridge12_final.predict(X_hold_linear_all[:, final_mask12]))

# Nonlinear challenger.
final_tree_pre = build_tree_preprocessor(location_mode='both')
X_dev_tree_all = final_tree_pre.fit_transform(X_dev_raw)
X_hold_tree_all = final_tree_pre.transform(X_hold_raw)
final_tree_names = np.asarray(final_tree_pre.get_feature_names_out(), dtype=str)
final_tree_groups = np.asarray([feature_group(n) for n in final_tree_names])
final_tree_mask9 = np.isin(final_tree_groups, sorted(SELECTED_GROUPS))
X_dev_hgb = dense_if_needed(X_dev_tree_all[:, final_tree_mask9])
X_hold_hgb = dense_if_needed(X_hold_tree_all[:, final_tree_mask9])
hgb_final = HistGradientBoostingRegressor(
    max_iter=140, learning_rate=0.06, max_leaf_nodes=15,
    min_samples_leaf=20, l2_regularization=1.0,
    loss='squared_error', random_state=RANDOM_STATE,
).fit(X_dev_hgb, np.log1p(y_dev))
pred_hgb = np.expm1(hgb_final.predict(X_hold_hgb))

holdout_predictions_before_target = pd.DataFrame({
    'meta_source_row_id': canonical_holdout_features['meta_source_row_id'].to_numpy(),
    'meta_sale_date': canonical_holdout_features['meta_sale_date'].dt.strftime('%Y-%m-%d'),
    'huber_log_9': pred_huber,
    'ridge_log_9': pred_ridge9,
    'ridge_log_full12': pred_ridge12,
    'hist_gradient_boosting_log_9': pred_hgb,
})
holdout_predictions_before_target.to_csv(TABLE_DIR / '28_holdout_predictions_before_target_open.csv', index=False)
display(holdout_predictions_before_target.head())

## 5.2 One-time holdout evaluation

The cell below now maps the frozen holdout row IDs back to their actual prices and evaluates the already-saved predictions. No model is refitted after these metrics are viewed.

In [ ]:
# ONE-TIME HOLDOUT TARGET OPEN
holdout_labels = pd.read_csv(LOCKED_HOLDOUT_LABEL_PATH).rename(
    columns={'source_row_id': 'meta_source_row_id'}
)

holdout_evaluation = holdout_predictions_before_target.merge(
    holdout_labels, on='meta_source_row_id', validate='one_to_one'
)
model_prediction_columns = [
    'huber_log_9', 'ridge_log_9', 'ridge_log_full12', 'hist_gradient_boosting_log_9'
]
final_holdout_rows = []
for model_name in model_prediction_columns:
    final_holdout_rows.append({'model': model_name, **regression_metrics(holdout_evaluation['price'], holdout_evaluation[model_name])})
final_holdout_metrics = pd.DataFrame(final_holdout_rows).sort_values('mae').reset_index(drop=True)
final_holdout_metrics.insert(0, 'mae_rank', np.arange(1, len(final_holdout_metrics) + 1))
final_holdout_metrics.to_csv(TABLE_DIR / '29_final_holdout_metrics.csv', index=False)
display(final_holdout_metrics)

fig, ax = plt.subplots(figsize=(9, 5))
ranked = final_holdout_metrics.sort_values('mae')
ax.barh(ranked['model'], ranked['mae'])
ax.invert_yaxis()
ax.set(title='One-time future-holdout MAE', xlabel='MAE ($)')
fig.tight_layout()
fig.savefig(FIG_DIR / '08_holdout_model_mae.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Final Huber residual diagnostics, calibration, and price segments.
holdout_evaluation['huber_residual'] = holdout_evaluation['huber_log_9'] - holdout_evaluation['price']
holdout_evaluation['huber_absolute_error'] = np.abs(holdout_evaluation['huber_residual'])

# Frozen development thresholds.
p90_dev = float(canonical_development['price'].quantile(0.90))
p99_dev = float(canonical_development['price'].quantile(0.99))
holdout_evaluation['price_segment'] = np.select(
    [holdout_evaluation['price'] > p99_dev, holdout_evaluation['price'] > p90_dev],
    ['above_development_p99', 'above_development_p90'],
    default='at_or_below_development_p90',
)
segment_rows = []
for segment, group in holdout_evaluation.groupby('price_segment'):
    segment_rows.append({'segment': segment, 'n': len(group), **regression_metrics(group['price'], group['huber_log_9'])})
holdout_segment_metrics = pd.DataFrame(segment_rows)
holdout_segment_metrics.to_csv(TABLE_DIR / '30_holdout_price_segment_metrics.csv', index=False)
display(holdout_segment_metrics)

# Sensitivity to the single largest actual price; official metrics retain it.
largest_index = holdout_evaluation['price'].idxmax()
sensitivity = pd.DataFrame([
    {'scenario': 'official_all_holdout_rows', 'n': len(holdout_evaluation), **regression_metrics(holdout_evaluation['price'], holdout_evaluation['huber_log_9'])},
    {'scenario': 'sensitivity_excluding_largest_actual_price', 'n': len(holdout_evaluation) - 1, **regression_metrics(
        holdout_evaluation.drop(index=largest_index)['price'],
        holdout_evaluation.drop(index=largest_index)['huber_log_9'],
    )},
])
sensitivity.to_csv(TABLE_DIR / '31_extreme_record_sensitivity.csv', index=False)
display(sensitivity)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(np.log1p(holdout_evaluation['price']), np.log1p(holdout_evaluation['huber_log_9']), alpha=0.5, s=18)
limits = [
    min(np.log1p(holdout_evaluation['price']).min(), np.log1p(holdout_evaluation['huber_log_9']).min()),
    max(np.log1p(holdout_evaluation['price']).max(), np.log1p(holdout_evaluation['huber_log_9']).max()),
]
ax.plot(limits, limits, linestyle='--')
ax.set(title='Frozen Huber: predicted versus actual', xlabel='log(1 + actual price)', ylabel='log(1 + predicted price)')
fig.tight_layout()
fig.savefig(FIG_DIR / '09_huber_predicted_vs_actual.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Date-cluster bootstrap confidence interval for primary-model holdout MAE.
rng = np.random.default_rng(RANDOM_STATE)
unique_dates = holdout_evaluation['meta_sale_date'].unique()
bootstrap_mae = []
actual_array = holdout_evaluation['price'].to_numpy(float)
pred_array = holdout_evaluation['huber_log_9'].to_numpy(float)
date_array = holdout_evaluation['meta_sale_date'].to_numpy()
for _ in range(N_BOOTSTRAPS):
    sampled_dates = rng.choice(unique_dates, size=len(unique_dates), replace=True)
    indices = np.concatenate([np.flatnonzero(date_array == d) for d in sampled_dates])
    bootstrap_mae.append(mean_absolute_error(actual_array[indices], pred_array[indices]))
bootstrap_ci = np.percentile(bootstrap_mae, [2.5, 97.5])
print(f'Date-cluster bootstrap 95% MAE interval: ${bootstrap_ci[0]:,.0f} to ${bootstrap_ci[1]:,.0f}')

## 5.3 Final recommendation

The frozen **log-target Huber regression** remains the preferred model because it has the lowest holdout MAE and median absolute error, robust behavior under influential transactions, and lower complexity than the nonlinear challengers.

Ridge remains a useful model-risk challenger. The principal weakness is the luxury tail, which should be routed for specialist review rather than treated as equally supported by the mainstream model.

# CRISP-DM Phase 6 — Deployment and Productionization

The model is now refitted using all 4,551 valid positive-price observations **after** the independent evaluation is permanently recorded. The frozen architecture is not changed.

In [ ]:
class SelectedGroupHuber(BaseEstimator, RegressorMixin):
    def __init__(self, epsilon=1.35, alpha=0.0001, max_iter=2000, tol=1e-5):
        self.epsilon = epsilon
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X: pd.DataFrame, y):
        self.preprocessor_ = build_linear_preprocessor(location_mode='both')
        transformed = self.preprocessor_.fit_transform(X)
        names = np.asarray(self.preprocessor_.get_feature_names_out(), dtype=str)
        groups = np.asarray([feature_group(n) for n in names])
        self.selected_mask_ = np.isin(groups, sorted(SELECTED_GROUPS))
        self.selected_feature_names_ = names[self.selected_mask_]
        self.model_ = HuberRegressor(
            epsilon=self.epsilon, alpha=self.alpha,
            max_iter=self.max_iter, tol=self.tol,
        ).fit(transformed[:, self.selected_mask_], np.log1p(np.asarray(y, dtype=float)))
        return self

    def predict(self, X: pd.DataFrame):
        transformed = self.preprocessor_.transform(X)
        prediction = np.expm1(self.model_.predict(transformed[:, self.selected_mask_]))
        return np.clip(prediction, 0, None)


class HousePriceServiceBundle:
    def fit(self, canonical: pd.DataFrame, y):
        self.regressor_ = SelectedGroupHuber().fit(canonical, y)
        self.reference_city_ = set(canonical['city'].astype(str))
        self.reference_zip_ = set(canonical['zip_code'].astype(str).str.zfill(5))

        structural = build_structural_features(canonical)
        self.structural_preprocessor_ = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler(quantile_range=(10, 90))),
        ])
        Xs = self.structural_preprocessor_.fit_transform(structural)
        self.clusterer_ = KMeans(n_clusters=3, n_init=25, random_state=RANDOM_STATE).fit(Xs)
        labels = self.clusterer_.labels_

        profile = canonical.assign(cluster_id=labels).groupby('cluster_id').agg(
            living=('sqft_living', 'median'), age=('property_age', 'median'),
            basement=('has_basement', 'mean'), floors=('floors', 'median'),
        )
        basement_cluster = int(profile['basement'].idxmax())
        compact_cluster = int(profile.drop(index=basement_cluster)['living'].idxmin())
        newer_cluster = int([i for i in profile.index if i not in {basement_cluster, compact_cluster}][0])
        self.cluster_name_map_ = {
            compact_cluster: 'Compact older non-basement',
            basement_cluster: 'Older basement-oriented',
            newer_cluster: 'Newer larger multi-story',
        }

        self.iforest_ = IsolationForest(
            n_estimators=N_IFOREST_TREES, max_samples=min(1024, len(canonical)),
            contamination='auto', random_state=RANDOM_STATE, n_jobs=-1,
        ).fit(Xs)
        self.iforest_threshold_ = float(np.quantile(self.iforest_.decision_function(Xs), 0.01))
        self.mcd_ = MinCovDet(support_fraction=0.75, random_state=RANDOM_STATE).fit(Xs)
        self.mahalanobis_threshold_ = float(np.quantile(self.mcd_.mahalanobis(Xs), 0.99))
        self.predicted_luxury_threshold_ = 895000.0
        self.predicted_extreme_threshold_ = 2000000.0
        self.training_n_ = len(canonical)
        return self

    def predict_canonical(self, canonical: pd.DataFrame) -> pd.DataFrame:
        prediction = self.regressor_.predict(canonical)
        Xs = self.structural_preprocessor_.transform(build_structural_features(canonical))
        cluster_id = self.clusterer_.predict(Xs)
        if_score = self.iforest_.decision_function(Xs)
        md2 = self.mcd_.mahalanobis(Xs)
        novelty_if = if_score <= self.iforest_threshold_
        novelty_md = md2 >= self.mahalanobis_threshold_
        unseen_city = ~canonical['city'].astype(str).isin(self.reference_city_)
        unseen_zip = ~canonical['zip_code'].astype(str).str.zfill(5).isin(self.reference_zip_)
        data_quality = canonical['room_count_invalid'].fillna(0).astype(int).eq(1)
        luxury = prediction >= self.predicted_luxury_threshold_
        extreme = prediction >= self.predicted_extreme_threshold_

        output = pd.DataFrame(index=canonical.index)
        output['predicted_price'] = prediction
        output['structural_segment'] = [self.cluster_name_map_[int(c)] for c in cluster_id]
        output['novelty_isolation_forest'] = novelty_if
        output['novelty_robust_distance'] = novelty_md
        output['novelty_union'] = novelty_if | novelty_md
        output['novelty_consensus'] = novelty_if & novelty_md
        output['unseen_city'] = unseen_city.to_numpy()
        output['unseen_zip'] = unseen_zip.to_numpy()
        output['data_quality_flag'] = data_quality.to_numpy()
        output['predicted_luxury_flag'] = luxury
        output['predicted_extreme_luxury_flag'] = extreme

        reasons = []
        for i in range(len(output)):
            row_reasons = []
            if extreme[i]:
                row_reasons.append('predicted_extreme_luxury')
            elif luxury[i]:
                row_reasons.append('predicted_luxury')
            if output.iloc[i]['novelty_consensus']:
                row_reasons.append('structural_novelty_consensus')
            elif output.iloc[i]['novelty_union']:
                row_reasons.append('structural_novelty')
            if output.iloc[i]['unseen_city'] or output.iloc[i]['unseen_zip']:
                row_reasons.append('unseen_location')
            if output.iloc[i]['data_quality_flag']:
                row_reasons.append('input_data_quality')
            reasons.append('|'.join(row_reasons))
        output['review_reasons'] = reasons
        output['manual_review_required'] = output['review_reasons'].ne('')
        return output

    def predict_raw(self, raw: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(raw, pd.DataFrame):
            raise TypeError('raw must be a pandas DataFrame')
        prohibited = sorted({'price', 'price_per_sqft'} & set(raw.columns))
        if prohibited:
            raise ValueError(f'Remove target or target-derived fields before inference: {prohibited}')
        canonical, _ = canonicalize_source(raw, include_target=False)
        canonical['meta_sale_date'] = pd.to_datetime(canonical['meta_sale_date'])
        return self.predict_canonical(canonical)

In [ ]:
# Production refit on all valid labeled rows using the frozen architecture.
canonical_holdout_labeled = canonical_holdout_features.merge(
    holdout_labels, on='meta_source_row_id', how='inner', validate='one_to_one'
)
canonical_positive_all = pd.concat(
    [canonical_development, canonical_holdout_labeled], ignore_index=True
).sort_values(['meta_sale_date', 'meta_source_row_id']).reset_index(drop=True)

X_all_positive = canonical_positive_all.drop(columns=['price'])
y_all_positive = canonical_positive_all['price'].to_numpy(float)

production_service = HousePriceServiceBundle().fit(X_all_positive, y_all_positive)
MODEL_PATH = MODEL_DIR / 'house_price_huber_service.pkl'
with MODEL_PATH.open('wb') as handle:
    cloudpickle.dump(production_service, handle)

production_summary = {
    'training_rows': int(len(X_all_positive)),
    'selected_feature_count': int(len(production_service.regressor_.selected_feature_names_)),
    'model_path': str(MODEL_PATH),
    'serialization': 'cloudpickle for fresh-process portability',
    'historical_scope': 'Washington residential transactions, May-July 2014',
    'live_use_warning': 'Retrain on recent representative transactions before current-market deployment.',
}
with open(TABLE_DIR / '32_production_refit_summary.json', 'w') as handle:
    json.dump(production_summary, handle, indent=2)
print(json.dumps(production_summary, indent=2))

In [ ]:
# Example raw and canonical inference records.
example_canonical = canonical_holdout_features.iloc[[0]].copy()
example_canonical_output = production_service.predict_canonical(example_canonical)
display(example_canonical_output)
example_canonical_output.to_csv(TABLE_DIR / '33_example_canonical_inference_output.csv', index=False)

example_raw = holdout_raw_features_only.iloc[[0]].copy()
example_raw_output = production_service.predict_raw(example_raw)
display(example_raw_output)
example_raw.drop(columns=['sale_date'], errors='ignore').to_csv(
    TABLE_DIR / '34_example_raw_inference_input.csv', index=False
)
example_raw_output.to_csv(TABLE_DIR / '35_example_raw_inference_output.csv', index=False)

# Confirm that the serialized service loads and predicts in a clean Python process.
SMOKE_INPUT_PATH = TABLE_DIR / '34_example_raw_inference_input.csv'
smoke_code = f"""
from joblib.externals import cloudpickle
import pandas as pd
with open(r'{MODEL_PATH}', 'rb') as handle:
    service = cloudpickle.load(handle)
raw = pd.read_csv(r'{SMOKE_INPUT_PATH}')
out = service.predict_raw(raw)
assert len(out) == 1
assert out['predicted_price'].gt(0).all()
print(out[['predicted_price', 'manual_review_required']].to_string(index=False))
"""
subprocess.check_call([sys.executable, '-c', smoke_code])
print('Fresh-process model load and prediction: PASS')

## 6.1 Monitoring and retraining policy

Recommended monitoring layers:

- **Input quality:** schema failures, impossible area identities, invalid room counts
- **Population drift:** PSI, unseen city/ZIP share, structural cluster-mix shift, novelty rate
- **Performance drift:** MAE, median AE, WAPE, RMSLE, signed error, luxury-segment MAE

Operational PSI guide:

- PSI < 0.10: normal
- 0.10 ≤ PSI < 0.25: warning
- PSI ≥ 0.25: critical review

Retraining should combine scheduled quarterly review with event-driven triggers. A challenger should replace the incumbent only after a new chronological holdout demonstrates material improvement.

In [ ]:
monitoring_policy = pd.DataFrame([
    ('overall_mae', float(final_holdout_metrics.loc[final_holdout_metrics['model'].eq('huber_log_9'), 'mae'].iloc[0]), 1.25, 1.50),
    ('median_absolute_error', float(final_holdout_metrics.loc[final_holdout_metrics['model'].eq('huber_log_9'), 'median_ae'].iloc[0]), 1.25, 1.50),
    ('wape', float(final_holdout_metrics.loc[final_holdout_metrics['model'].eq('huber_log_9'), 'wape'].iloc[0]), 1.25, 1.50),
    ('rmsle', float(final_holdout_metrics.loc[final_holdout_metrics['model'].eq('huber_log_9'), 'rmsle'].iloc[0]), 1.25, 1.50),
], columns=['metric', 'reference', 'warning_multiplier', 'critical_multiplier'])
monitoring_policy['warning_threshold'] = monitoring_policy['reference'] * monitoring_policy['warning_multiplier']
monitoring_policy['critical_threshold'] = monitoring_policy['reference'] * monitoring_policy['critical_multiplier']
monitoring_policy.to_csv(TABLE_DIR / '34_monitoring_policy.csv', index=False)
display(monitoring_policy)

# Final conclusions

1. **Data discipline mattered more than algorithmic complexity.** Leakage prevention, temporal validation, robust cleaning, and carefully engineered location/size features generated the largest gains.
2. **The log target was consistently beneficial.** It reduced skewness and improved generalization across linear and tree families.
3. **Huber regression was the final recommendation.** It produced the best future-period MAE while remaining robust and interpretable.
4. **The luxury tail remains the primary risk.** High-value or structurally novel properties require manual review.
5. **The serialized model is historical.** It demonstrates a complete production workflow but must be retrained using recent market data before use for present-day valuations.

The notebook has now completed all six CRISP-DM phases:

\[
Business\ Understanding \rightarrow Data\ Understanding \rightarrow Data\ Preparation
\rightarrow Modeling \rightarrow Evaluation \rightarrow Deployment
\]

# Download the complete results package

The final cell saves the principal notebook outputs, tables, figures, and serialized production model into one ZIP archive.

In [ ]:
# Save a compact run manifest and package all generated artifacts.
run_manifest = {
    'data_path': str(DATA_PATH),
    'data_sha256': file_hash,
    'random_state': RANDOM_STATE,
    'fast_mode': FAST_MODE,
    'development_rows': int(len(canonical_development)),
    'holdout_rows': int(len(canonical_holdout_features)),
    'positive_labeled_rows_for_production_refit': int(len(canonical_positive_all)),
    'frozen_primary_model': FROZEN_PRIMARY_MODEL,
    'selected_groups': sorted(SELECTED_GROUPS),
    'model_serialization': 'cloudpickle',
}
with open(OUTPUT_DIR / 'run_manifest.json', 'w') as handle:
    json.dump(run_manifest, handle, indent=2)

ZIP_BASE = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}_results'
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('Results package:', zip_path)
print('Serialized model:', MODEL_PATH)

if AUTO_DOWNLOAD_RESULTS and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))